# Advanced · MCTS internals + training math

Deeper dive than `advanced_internals.ipynb`. Covers every non-obvious decision and number that goes into Orca's search and training, with file:line citations so you can read the actual source.

**Companion notebook:** `advanced_engine_and_search.ipynb` covers the C game engine, alpha-beta, transposition tables, threat search, endgame solver, opening book.

**Table of contents**

1. Dirichlet root noise (training-time exploration)
2. Temperature scheduling for move selection
3. C heuristic blending into the MCTS prior
4. Threat map blending into the policy head
5. Virtual loss for parallel MCTS
6. The eight network architectures
7. Cosine annealing learning rate schedule
8. Mixed precision (autocast + GradScaler)
9. Gradient clipping
10. Soft vs one-hot policy targets (auto-switch)
11. Temporal value decay on opponent-game samples
12. Replay buffer priority math
13. Progressive game-length filters
14. Tactical priority boosts (blocking, survival, hindsight)
15. AutoTuner rules in detail
16. Curriculum learning
17. Plateau detection and sim boost

In [ ]:
!pip install --quiet hexbot

## 1. Dirichlet root noise (training-time exploration)

If MCTS always picked the action with the highest PUCT score from the root, self-play games would be deterministic and the network would only ever see one opening. AlphaZero injects noise at the root prior so different games explore different lines:

$$\pi'(s_0, a) = (1 - \epsilon)\,\pi(s_0, a) + \epsilon\,\eta_a$$

where $\eta \sim \text{Dirichlet}(\alpha)$ is sampled fresh for each game.

Orca's settings (`orca/config.py`):

- $\alpha = 0.3$ (`DIRICHLET_ALPHA`): biases noise toward roughly-uniform draws on Hex's high branching factor.
- $\epsilon = 0.15$ (`DIRICHLET_EPSILON`): fraction of the prior replaced by noise. AlphaGo used 0.25; this is subtler.

Only the root gets noise. Children inherit the unnoised prior. Disable noise entirely by passing `add_noise=False` to `BatchedMCTS.search()`.

In [ ]:
import numpy as np
from orca.config import DIRICHLET_ALPHA, DIRICHLET_EPSILON

print(f"alpha   = {DIRICHLET_ALPHA}")
print(f"epsilon = {DIRICHLET_EPSILON}")

# What does a fresh Dirichlet sample look like over, say, 8 candidate moves?
for i in range(3):
    eta = np.random.dirichlet([DIRICHLET_ALPHA] * 8)
    bars = '  '.join(f'{p:.2f}' for p in eta)
    print(f'sample {i+1}: {bars}   (sum={eta.sum():.3f})')

## 2. Temperature scheduling for move selection

After MCTS returns visit counts $N(s, a)$, the actual move is sampled from:

$$P(a) \propto N(s, a)^{1/\tau}$$

- $\tau \to 0$: argmax (greedy)
- $\tau = 1$: proportional to visits
- $\tau > 1$: smoother, more exploratory

Orca uses two regimes (`orca/config.py:TEMP_THRESHOLD = 20`):

- First 20 moves: $\tau = 1$ (proportional sampling for opening diversity).
- After move 20: $\tau \to 0$ (greedy).

Implementation: `orca/search.py:293` does `visit_count ** (1.0 / temperature)`. For inference (`BatchedMCTS.search(game, temperature=0.15)`) the default is 0.15, which is nearly greedy but allows occasional second-best picks for game-to-game variety against SealBot.

In [ ]:
# Watch the distribution shape as temperature changes
visits = np.array([120, 60, 30, 12, 5, 3, 2, 1])

for tau in [0.1, 0.5, 1.0, 2.0]:
    scaled = visits ** (1.0 / tau)
    probs = scaled / scaled.sum()
    print(f"tau={tau:>4}: {' '.join(f'{p:.3f}' for p in probs)}")

## 3. C heuristic blending into the MCTS prior

The neural network's policy prior is good, but the C engine's cheap line-counting heuristic catches a few things the network sometimes misses (e.g. immediate tactical resources at the edge of the training distribution). Orca blends both:

$$\pi_\text{blended}(s, a) = (1 - w_a) \cdot \pi_\text{net}(s, a) + w_a \cdot \pi_\text{C}(s, a)$$

with weights that depend on the move's distance from existing stones (`orca/config.py`):

- `C_BLEND_ADJACENT = 0.15`: moves within 1 ring of a stone.
- `C_BLEND_DISTANT  = 0.05`: moves further out, where the C heuristic is less reliable.

This is a small but consistent strength boost early in training when the network's policy is still noisy.

## 4. Threat map blending into the policy head

Each of the 7 input channels carries information, but channels 5 and 6 (the threat maps) are special: cells that already have a 4-in-a-row latent line are tactically critical. The framework injects this signal directly into the policy logits before the final softmax:

$$\text{logits} \leftarrow \text{logits} + \lambda_\tau \cdot M_\tau(s)$$

where $M_\tau$ is a 19x19 spatial map and $\lambda_\tau$ = `THREAT_POLICY_BLEND = 0.5`.

Wired into every network variant:

- `orca/network.py:233`
- `orca/hex_conv.py:210, 314`
- `orca/hex_gnn.py:162`

Set `THREAT_POLICY_BLEND = 0` to disable for ablation studies.

## 5. Virtual loss for parallel MCTS

When multiple threads expand the tree concurrently (`BatchedMCTS` with workers), naive selection would have all threads walk to the same leaf. **Virtual loss** prevents this: as a thread reserves a node for evaluation it temporarily increments the visit count by a loss-shaped value, making that node look worse to other threads until the real evaluation comes back and replaces it.

In practice this lets `BatchedMCTS` collect up to `MCTS_BATCH_SIZE = 64` distinct leaves per network forward pass without any of them being duplicates. Throughput on a GPU scales nearly linearly with batch size up to the tensor core limit.

## 6. The eight network architectures

All registered via `create_network(config_name)`. Their shared structure: 7-channel input, three heads (policy 361 + value 1 + threat 4). They differ in trunk:

| `--config` | Params | Trunk | When to pick |
|---|---:|---|---|
| `fast`              | 656K  | 3 ResBlocks @ 64 filters | Quick experiments, weak hardware |
| `standard`          | 3.9M  | 12 ResBlocks @ 128 filters | Default, balanced |
| `hex-masked`        | 3.9M  | 3x3 conv with non-hex neighbours zeroed | **Recommended** when you have a GPU |
| `hex-native`        | 3.1M  | True 7-weight hex kernel, no masking | Cleanest hex inductive bias |
| `hex-native-circular` | 3.1M | Same + toroidal padding | Sharper boundary handling |
| `large`             | 14.5M | 12 ResBlocks @ 256 filters | Max strength, slow training |
| `orca-transformer`  | 4.4M  | CNN + 2-layer transformer attention | Experimental |
| `hex-gnn`           | 432K  | GNN over hex topology | Experimental, tiny model |
| `multiscale`        | 1.1M  | Local CNN + global attention two-tower | Experimental |

Threat blending (section 4) applies to all of them. See `orca/network.py`, `orca/hex_conv.py`, `orca/hex_gnn.py`, `orca/transformer_net.py`, `orca/multiscale_net.py` for the actual classes.

In [ ]:
from hexbot import create_network
import torch

for cfg in ['fast', 'standard', 'hex-masked', 'large']:
    net = create_network(cfg)
    params = sum(p.numel() for p in net.parameters())
    out = net(torch.randn(1, 7, 19, 19))
    print(f"{cfg:>14}: {params:>10,} params  policy={tuple(out[0].shape)} value={tuple(out[1].shape)} threat={tuple(out[2].shape)}")

## 7. Cosine annealing learning rate schedule

Orca uses `torch.optim.lr_scheduler.CosineAnnealingWarmRestarts` over training iterations:

$$\eta_t = \eta_\min + \tfrac{1}{2}(\eta_\max - \eta_\min)\left(1 + \cos\left(\frac{T_{cur}}{T_i}\pi\right)\right)$$

Period $T_i$ doubles after each restart (`T_mult=2`). The intuition: smooth decay within a period, then jump back near the initial LR to escape sharp minima.

Disable with `--no-adaptive-lr` to use a fixed `LEARNING_RATE = 0.001`.

In [ ]:
# Simulate the schedule's shape
import torch

params = [torch.zeros(1, requires_grad=True)]
opt = torch.optim.Adam(params, lr=0.001)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2, eta_min=1e-5)
lrs = []
for _ in range(80):
    lrs.append(opt.param_groups[0]['lr'])
    opt.step()
    sched.step()

# ASCII-plot
max_lr = max(lrs)
for i, lr in enumerate(lrs):
    bar = '#' * int(40 * lr / max_lr)
    if i % 4 == 0:
        print(f"iter {i:>3}: {lr:.5f}  {bar}")

## 8. Mixed precision (autocast + GradScaler)

On NVIDIA tensor cores, FP16 matmul is roughly 2x faster than FP32 with minimal accuracy cost. Two pieces:

**`autocast`** wraps the forward + loss computation. Layers like `Conv2d` and `Linear` run in FP16; operations that need precision (e.g. `BatchNorm` running stats, the final softmax) stay in FP32. PyTorch picks per-op.

**`GradScaler`** solves the FP16 underflow problem. Small gradient values round to zero in FP16. The scaler multiplies the loss by a large scale factor before backward, then unscales the gradients before the optimizer step:

$$\text{loss}_{scaled} = s \cdot \text{loss}, \quad g_{scaled} = s \cdot g, \quad g_{unscaled} = g_{scaled} / s$$

If any gradient is non-finite (overflowed), the optimizer step is skipped and `s` is halved on the next iteration. The scaler dynamically adapts.

Wiring: `orca/train.py:978` instantiates `torch.amp.GradScaler('cuda')`. MPS path silently disables mixed precision because PyTorch's FP16 support on Metal is unreliable.

## 9. Gradient clipping

After the backward pass, the global norm of all gradients is clipped to `GRAD_CLIP_NORM = 1.0`:

$$g_i \leftarrow g_i \cdot \min\!\left(1, \frac{c}{\|g\|_2}\right)$$

Why it matters in AlphaZero: the replay buffer holds samples from many policy generations, and an unusually large gradient on an outlier sample can destabilize the trunk. Clipping caps the per-step damage. Set `GRAD_CLIP_NORM = 0` in config to disable.

## 10. Soft vs one-hot policy targets (auto-switch)

On Orca's moves during a SealBot game, the policy target is normally the soft MCTS visit distribution. But on a fresh, random-init network the MCTS distribution is meaningless and learning from it actively hurts. The trainer auto-switches:

- If `_last_policy_loss >= 3.0`: use a one-hot target (which move was actually played). Safe but coarse.
- If `_last_policy_loss < 3.0`: use the soft MCTS distribution. Higher signal.

The threshold (3.0) is hardcoded in `orca/train.py` near the SealBot game path. The same threshold gates whether SealBot's own moves are added as expert demonstrations (see commit `a68fff7`).

## 11. Temporal value decay on opponent-game samples

In games against SealBot, every sample collected before the end gets the game's final outcome as its value target. But a position 100 moves before the end is barely related to the outcome. Naive labelling would teach the value head that all those mid-game positions are 'lost' just because the final position was lost.

Decay solves this:

$$v_\text{target}(s_t) = \gamma^{T - t} \cdot z$$

with $\gamma = 0.99$ and $T - t$ the number of moves from $s_t$ to game end. Positions closest to the loss carry the strongest negative signal; mid-game positions get a weaker pull. Implemented in `_play_vs_ramora_batch` in `orca/train.py`.

Self-play games do not use decay; they use the standard AlphaZero `z = ±1` target.

## 12. Replay buffer priority math

Each sample carries a priority $p_i$. Sampling for the next minibatch is proportional:

$$P(i) = \frac{p_i^\alpha}{\sum_j p_j^\alpha}$$

with $\alpha = 1$ in Orca (linear priority sampling; no Boltzmann tempering). After each gradient step, the priority is updated based on TD-error:

$$p_i \leftarrow |z_i - v_\theta(s_i)| + \epsilon$$

with small $\epsilon$ to keep low-error samples eligible. The math is standard prioritised experience replay. What is hexbot-specific is the *initial* priority assignment, covered in the next two sections.

## 13. Progressive game-length filters

Very short games are mostly random; very long games tend to be hard-fought. The framework rewards length:

| Game length | Initial priority |
|---:|---:|
| < 10 moves | discarded entirely (junk) |
| 10-19 | 0.2 |
| 20-29 | 0.4 |
| 30-39 | 0.7 |
| 40-44 | 1.0 |
| 45-59 | 1.3 |
| 60+ | 1.8 |

Combined with priority sampling, this means a 60-move game contributes 9x more learning signal per move than a 20-move one. See `orca/data.py`.

## 14. Tactical priority boosts

Three orthogonal boosts (config values in `orca/config.py`):

- `BLOCKING_PRIORITY_BOOST = 5.0`: applied to a move that successfully blocked an opponent threat. Multiplied by 1.7x on extreme blocks (2+ simultaneous threats), 1.2x on critical hindsight detection.
- `SURVIVAL_PRIORITY_BOOST = 3.0`: applied to a move that survived a turn where the opponent had a winning threat available.
- **Recency weighting**: late-game moves get priority `1.0 + 2.0 * recency`, where `recency` is the move's index normalised by game length. Endgame moves matter more than openings.

The hindsight pass (in `_play_vs_ramora_batch`) replays each game with the actual moves and checks: "after Orca's move, did SealBot create 2+ threats?" If yes, the preceding Orca move was critical (it failed to defuse a forking position) and gets boosted retroactively.

## 15. AutoTuner rules in detail

`AutoTuner` (`orca/train.py`) observes per-iteration metrics and adjusts a handful of training knobs. The rules are explicit (no learned ML), which keeps debugging easy. Each iteration:

1. **MCTS sims cap**: hard cap at 50 during training. Higher sims produce slightly better data but blow up the iteration time budget.
2. **Game mix**: 60% normal positions, 15% endgame positions, 15% formation positions, 5% catalog positions, 5% sequence positions. Tactical positions are essential from iteration 0 so the network sees blocking and pattern recognition early.
3. **Hint blend decay**: starts at 0.3, decays at `-0.015 * iteration`, floors at 0. The C heuristic helps early; once the network is good, raw network policy is preferred.
4. **Train steps escalation**: if loss is decreasing over the last 3 iterations AND buffer is >90% full, bump train steps by 50 (capped at 300). Bigger network can absorb more updates per iteration once data quality is high.

Preview the rules without applying them: `--auto-tuner-dry-run`.

In [ ]:
from orca.train import AutoTuner

at = AutoTuner(dry_run=False)
print('initial params:')
for k, v in at.params.items():
    print(f"  {k}: {v}")

# Simulate a few iterations with decreasing loss + full buffer
for i in range(5):
    at.update({'total_loss': 10 - i*2, 'buffer_fill': 0.95, 'elo': 1000 + i*30}, iteration=i)

print('\nafter 5 iters of improving metrics:')
for k, v in at.params.items():
    print(f"  {k}: {v}")

## 16. Curriculum learning

Six skill levels in `orca/curriculum.py`. Each level changes the difficulty of self-play positions:

| Level | sims | games | Description |
|---:|---:|---:|---|
| 1 | 30 | 8 | Basics: place, block, simple lines |
| 2 | 60 | 12 | Open 5s, simple threats |
| 3 | 100 | 20 | Forks, double threats |
| 4 | 150 | 30 | Endgame, deep tactics |
| 5 | 200 | 40 | Strategic play |
| 6 | 250 | 50 | Mature, all features active |

Level advances when ELO crosses configured thresholds; disabled with `--no-curriculum`.

In [ ]:
from orca.curriculum import SkillCurriculum

c = SkillCurriculum()
print(f'starting at level {c.level}, sims={c.get_config()["sims"]}, games={c.get_config()["games"]}')

## 17. Plateau detection and sim boost

When ELO stops moving over a window of iterations, training gets stuck. The plateau machinery (recently wired into the curriculum, see commit `c6bd82c`) reacts:

- `PLATEAU_THRESHOLD = 15`: ELO delta below this counts as 'stalled'.
- `PLATEAU_ITERS = 10`: consecutive stalls before declaring a plateau.
- `PLATEAU_SIM_BOOST = 50`: extra MCTS sims to throw at the problem until ELO moves again (capped at 400 total).

The boost trades wall time for better-quality samples. Once ELO starts moving again, the sim count returns to the curriculum default.

In [ ]:
import orca.train as t

# Reset module state
t._curriculum_last_elo = None
t._curriculum_stall_iters = 0

# Simulate 12 iterations of stalled ELO (delta < 15)
for i in range(12):
    stall = t.update_curriculum_plateau(1000.0 + i * 2, threshold=15.0)
print(f'after 12 stalled iters: stall_iters = {stall}')

sims_normal = t.get_curriculum_sims(50, configured_sims=100, plateau_iters=0, plateau_boost=0)
sims_boost  = t.get_curriculum_sims(50, configured_sims=100, plateau_iters=10, plateau_boost=50)
print(f'sims without boost: {sims_normal}')
print(f'sims with boost:    {sims_boost}  (plateau detected)')

## Where to go next

- `advanced_engine_and_search.ipynb`: C engine internals, alpha-beta with TT/killers/LMR, threat search, endgame solver, opening book, GPU inference server.
- `advanced_internals.ipynb`: the gentler overview that gathers everything in one place.
- `orca/config.py`: every single constant in one file, with comments on what each one controls.